In [1]:
import numpy as np

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score
)

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
X = np.load("../processed/X_offset80.npy")
y = np.load("../processed/y_offset80.npy")
subjects = np.load("../processed/subjects_offset80.npy")
print(X.shape)
print(y.shape)
print(subjects.shape)

(4635, 7, 297)
(4635,)
(4635,)


In [3]:
print(len(np.unique(subjects)))
print(np.unique(subjects))

103
[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  44  45  46  47  48  49  50  51  52  53  54  55
  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72  73
  74  75  76  77  78  79  80  81  82  83  84  85  86  87  90  91  93  94
  95  96  97  98  99 101 102 103 105 106 107 108 109]


In [4]:
unique_subjects = np.unique(subjects)

subject_kfold = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [5]:
def build_model():

    inputs = Input(shape=(7,297))

    x = LSTM(
        256,
        return_sequences=True
    )(inputs)

    x = Dropout(0.2)(x)

    x = LSTM(
        256,
        return_sequences=True
    )(x)

    x = Dropout(0.1)(x)

    x = LSTM(
        256
    )(x)

    x = Dropout(0.2)(x)

    outputs = Dense(
        1,
        activation="sigmoid"
    )(x)

    model = Model(
        inputs,
        outputs
    )

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [6]:
# Fold 1 only

train_sub_idx, test_sub_idx = next(
    subject_kfold.split(unique_subjects)
)

train_subjects = unique_subjects[
    train_sub_idx
]

test_subjects = unique_subjects[
    test_sub_idx
]

print("Train Subjects:", len(train_subjects))
print("Test Subjects :", len(test_subjects))

print(
    "Intersection:",
    np.intersect1d(
        train_subjects,
        test_subjects
    )
)

train_mask = np.isin(
    subjects,
    train_subjects
)

test_mask = np.isin(
    subjects,
    test_subjects
)

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print("\nTrain Shape:")
print(X_train.shape)

print("\nTest Shape:")
print(X_test.shape)

print("\nTrain Labels:")
print(
    np.unique(
        y_train,
        return_counts=True
    )
)

print("\nTest Labels:")
print(
    np.unique(
        y_test,
        return_counts=True
    )
)

Train Subjects: 92
Test Subjects : 11
Intersection: []

Train Shape:
(4140, 7, 297)

Test Shape:
(495, 7, 297)

Train Labels:
(array([0, 1]), array([2083, 2057]))

Test Labels:
(array([0, 1]), array([254, 241]))


In [7]:
scaler = StandardScaler()

X_train_flat = X_train.reshape(-1,297)
X_test_flat = X_test.reshape(-1,297)

X_train_flat = scaler.fit_transform(
    X_train_flat
)

X_test_flat = scaler.transform(
    X_test_flat
)

X_train = X_train_flat.reshape(
    X_train.shape
)

X_test = X_test_flat.reshape(
    X_test.shape
)

In [8]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

model = build_model()

history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 27s 180ms/step - accuracy: 0.5896 - loss: 0.6608 - val_accuracy: 0.6787 - val_loss: 0.6584
Epoch 2/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 36s 132ms/step - accuracy: 0.7214 - loss: 0.5419 - val_accuracy: 0.6884 - val_loss: 0.5661
Epoch 3/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 15s 127ms/step - accuracy: 0.7772 - loss: 0.4560 - val_accuracy: 0.6908 - val_loss: 0.6741
Epoch 4/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 17s 147ms/step - accuracy: 0.8288 - loss: 0.3692 - val_accuracy: 0.6981 - val_loss: 0.6828
Epoch 5/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 19s 130ms/step - accuracy: 0.8739 - loss: 0.2873 - val_accuracy: 0.6691 - val_loss: 0.8578
Epoch 6/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 66ms/step - accuracy: 0.9047 - loss: 0.2228 - val_accuracy: 0.6546 - val_loss: 1.0187
Epoch 7/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 66ms/step - accuracy: 0.9396 - loss: 0.1597 - val_accuracy: 0.6594 - val_loss: 1.3306


In [9]:
from sklearn.metrics import classification_report

pred = model.predict(X_test)

pred = (pred > 0.5).astype(int)

print(
    classification_report(
        y_test,
        pred
    )
)

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step
              precision    recall  f1-score   support

           0       0.65      0.85      0.74       254
           1       0.77      0.52      0.62       241

    accuracy                           0.69       495
   macro avg       0.71      0.68      0.68       495
weighted avg       0.71      0.69      0.68       495

